In [1]:
import numpy as np
import networkx as nx
from typing import List
import sys
from test_multicommodity import test_main

In [2]:
def read_coordinate(path="../data/input/coord.dat"):
    with open(path) as f:
        return [tuple(map(lambda x: round(float(x) * 25), line.split()[:2])) for line in f if line.strip()]


def create_raw_forcing(path="../data/input/rhs.dat"):
    def rhs(forcing, rho: float=0.0):
        g_forcing = forcing - rho * (forcing - np.mean(forcing))
        num_node = len(forcing)
        forcing_mat = np.zeros((num_node, num_node))
        for n in range(num_node):
            sum_g = np.sum(g_forcing) - g_forcing[n]
            sum_g -= g_forcing[n]
            coeff_r = g_forcing / sum_g
            forcing_mat[:, n] = -coeff_r * g_forcing[n]
            forcing_mat[n, n] = g_forcing[n]
        return forcing_mat
    
    with open(path) as forcing_f:
        lines = forcing_f.readlines()
        ncomm = len(lines)
        temp_forcing = np.zeros((ncomm, 3))
        for i, line in enumerate(lines):
            temp_forcing[i, :] = np.array([int(float(element)) for element in line.strip().split(" ")])
        return temp_forcing[:, :-1]

def rhs_construction(forcing, rho=0):
    g_forcing = forcing - rho * (forcing - np.mean(forcing))
    num_node = len(forcing)
    raw_forcing = np.zeros((num_node, num_node))
    for n in range(num_node):
        sum_g = np.sum(g_forcing)
        sum_g -= g_forcing[n]
        coeff_r = g_forcing / sum_g
        raw_forcing[:, n] = -coeff_r * g_forcing[n]
        raw_forcing[n, n] = g_forcing[n]
    return raw_forcing


coordinates = read_coordinate()
forcing = create_raw_forcing()
raw_forcing_mat = rhs_construction(forcing[:, 1])

In [3]:
def idx(coord):
    return coord[0] * 25 + coord[1]

In [4]:
mesh = nx.grid_2d_graph(25, 25)

In [5]:
forcing = np.zeros((625, 625))
num_node = len(coordinates)
for i in range(num_node):
    for j in range(num_node):
        forcing[idx(coordinates[i])][idx(coordinates[j])] = raw_forcing_mat[i][j]

In [6]:
edge_lists = np.array([list((idx(e[0]), idx(e[1]))) for e in mesh.edges()])
weights = np.ones_like(edge_lists[:,0])
forcing_ = [forcing[:, i] / 10000 for i in range(forcing.shape[1])]

In [7]:
potentials, conductivity_admk = test_main(edge_lists, weights, len(forcing_), forcing_, beta=1, max_iter=10)

Rhs236 is not balanced 9.0E-01
Rhs261 is not balanced 9.0E-01
Rhs262 is not balanced 9.0E-01
Rhs263 is not balanced 9.0E-01
Rhs264 is not balanced 9.1E-01
Rhs285 is not balanced 9.0E-01
Rhs286 is not balanced 9.0E-01
Rhs287 is not balanced 9.0E-01
Rhs288 is not balanced 9.0E-01
Rhs289 is not balanced 9.0E-01
Rhs290 is not balanced 9.0E-01
Rhs311 is not balanced 9.0E-01
Rhs312 is not balanced 9.0E-01
Rhs313 is not balanced 9.0E-01
Rhs314 is not balanced 9.0E-01
Rhs315 is not balanced 9.0E-01
Rhs334 is not balanced 9.0E-01
Rhs335 is not balanced 9.0E-01
Rhs336 is not balanced 9.0E-01
Rhs337 is not balanced 9.0E-01
Rhs338 is not balanced 9.0E-01
Rhs339 is not balanced 9.0E-01
Rhs340 is not balanced 9.0E-01
Rhs341 is not balanced 9.0E-01
Rhs360 is not balanced 9.0E-01
Rhs361 is not balanced 9.0E-01
Rhs362 is not balanced 9.0E-01
Rhs363 is not balanced 9.0E-01
Rhs364 is not balanced 9.0E-01
Rhs365 is not balanced 9.0E-01
Rhs383 is not balanced 9.0E-01
Rhs384 is not balanced 9.0E-01
Rhs385 i

/home/oreki/admk_thesis/tests/../src/admk/admk_graph.py:136: RuntimeWarning: invalid value encountered in scalar divide
  balance = np.sum(self.rhs[begin:end])/np.linalg.norm(self.rhs[begin:end])


ierr=0 avg_it=0000 max(res)=1.0e-09 max(pres)=9.9e-07


KeyboardInterrupt: 

In [ ]:
nx.draw_networkx_edges(mesh, {n: n for n in mesh.nodes()}, width=conductivity_admk / np.linalg.norm(conductivity_admk) * 50,  edge_color='C0', style='solid')